In [1]:
import mediapipe as mp
import cv2
import numpy as np
import time
import threading

BaseOptions = mp.tasks.BaseOptions
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

model_path = "face_landmarker.task"

latest_result = None
result_lock = threading.Lock()

def store_result(result, output_image, timestamp_ms):
    global latest_result
    with result_lock:
        latest_result = result

options = FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=model_path),
    running_mode=VisionRunningMode.LIVE_STREAM,
    result_callback=store_result
)


In [ ]:
cap = cv2.VideoCapture(0)
start_time = time.time()

time.sleep(1)

with FaceLandmarker.create_from_options(options) as landmarker:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        timestamp_ms = int((time.time() - start_time) * 1000)
        landmarker.detect_async(mp_image, timestamp_ms)

        with result_lock:
            current_result = latest_result

        if current_result and current_result.face_landmarks:
            landmarks = current_result.face_landmarks[0]
            h, w, _ = frame.shape

            for lm in landmarks:
                x, y = int(lm.x * w), int(lm.y * h)
                cv2.circle(frame, (x, y), 1, (0, 255, 0), -1)

    
            regions = {
                "forehead_ids": [109, 10, 338, 336, 9, 107],
                "left_cheek_ids": [116,111,117,118, 119, 120,100,142,36,205,123],
                "right_cheek_ids": [371,329,349,348,347,346,340,345,352,425,266]
            }


            region_points = {}
            for region_name, region in regions.items():
                points = np.array([
                    (int(landmarks[i].x * w), int(landmarks[i].y * h))
                    for i in region
                ])
                region_points[region_name] = points
                cv2.polylines(frame, [points], isClosed=True, color=(0,0,255) , thickness=2)

            mask = np.zeros(frame.shape[:2], dtype=np.uint8)
            for region_name, points in region_points.items():
                cv2.fillPoly(mask, [points], 255)

            mean_bgr = cv2.mean(frame, mask=mask)[:3]
            print(f"B: {mean_bgr[0]:.1f}, G: {mean_bgr[1]:.1f}, R: {mean_bgr[2]:.1f}")

        cv2.imshow("rPPG", frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

I0000 00:00:1781247701.060281 17147018 init-domain.cc:128] Fiber init: default domain = pthread, concurrency = 12, prefix = pthread-default
W0000 00:00:1781247701.062995 17147018 face_landmarker_graph.cc:180] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
I0000 00:00:1781247701.089057 17147018 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M3 Pro
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1781247701.094750 17147026 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1781247701.104791 17147026 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


B: 61.7, G: 72.1, R: 120.0
B: 60.0, G: 70.5, R: 116.6
B: 60.7, G: 70.8, R: 116.4
B: 60.6, G: 70.8, R: 116.6
B: 61.3, G: 71.4, R: 117.4
B: 61.8, G: 72.1, R: 118.3
B: 63.3, G: 73.4, R: 120.0
B: 63.7, G: 73.7, R: 120.5
B: 63.9, G: 74.0, R: 120.6
B: 63.9, G: 74.2, R: 120.9
B: 64.3, G: 74.3, R: 121.1
B: 64.4, G: 74.4, R: 121.4
B: 64.8, G: 74.6, R: 121.6
B: 65.0, G: 74.6, R: 121.9
B: 64.9, G: 74.7, R: 122.0
B: 65.0, G: 74.8, R: 122.1
B: 65.0, G: 74.9, R: 122.3
B: 65.1, G: 74.9, R: 122.5
B: 64.9, G: 74.9, R: 122.5
B: 64.9, G: 74.9, R: 122.3
B: 65.1, G: 74.9, R: 122.3
B: 64.9, G: 74.8, R: 122.0
B: 65.0, G: 74.9, R: 122.1
B: 65.0, G: 75.0, R: 122.4
B: 65.2, G: 75.0, R: 122.6
B: 65.0, G: 75.3, R: 122.2
B: 65.2, G: 75.3, R: 122.8
B: 64.8, G: 75.5, R: 122.9
B: 65.3, G: 75.2, R: 122.9
B: 65.0, G: 75.3, R: 123.0
B: 65.5, G: 75.2, R: 123.6
B: 65.1, G: 74.9, R: 123.5
B: 64.5, G: 74.0, R: 122.7
B: 63.9, G: 73.3, R: 121.6
B: 63.0, G: 72.8, R: 121.1
B: 62.0, G: 72.1, R: 120.6
B: 61.8, G: 72.0, R: 119.9
B

KeyboardInterrupt: 